# Failure Case Testing & Performance Degradation Analysis

This notebook evaluates the RAG system's robustness against common failure vectors:
1. **Out-of-Domain (OOD) Questions**: The system must reject answering questions unrelated to our financial filings corpus.
2. **Ambiguous Questions**: The system must request clarification rather than guessing context or hallucinating details.
3. **Multi-Hop Questions**: Measures the quality and latency degradation when querying queries that require aggregating facts across multiple documents.

In [ ]:
import os
import sys
# Ensure path is corrected to import project modules
sys.path.append(os.path.abspath('..'))

from backend.rag_pipeline import RAGPipeline
pipeline = RAGPipeline()
print("RAG Pipeline loaded successfully.")

## 1. Out-of-Domain (OOD) Questions

We query the RAG pipeline with questions that have no relevance to financial reports. The expected behavior is a clean refusal: "Insufficient evidence found."

In [ ]:
ood_queries = [
    "What is the capital of France?",
    "Who won the 2022 FIFA World Cup?",
    "Explain the chemical composition of table salt."
]

for query in ood_queries:
    print(f"Query: '{query}'")
    result = pipeline.query(query)
    print(f"Answer: {result['answer']}")
    print(f"Chunks Retrieved: {len(result['retrieved_chunks'])}")
    print("-" * 40)

## 2. Ambiguous Questions

Querying the RAG pipeline with vague keywords (e.g. "Revenue growth"). The expected behavior is that the pipeline detects the ambiguity and requests clarification.

In [ ]:
ambiguous_queries = [
    "Revenue growth",
    "Nvidia profit",
    "Risk factors"
]

for query in ambiguous_queries:
    print(f"Query: '{query}'")
    result = pipeline.query(query)
    print(f"Answer: {result['answer']}")
    print("-" * 40)

## 3. Multi-Hop Questions

Multi-hop queries require retrieving, fusing, and reasoning over multiple documents (e.g. comparing two different companies across multiple years). We will measure latency and retrieval degradation compared to single-fact lookups.

In [ ]:
single_hop = "What is Apple's revenue in 2024?"
multi_hop = "Compare risk factors of Nvidia and AMD over the last 3 years."

print("=== Running Single Hop lookup ===")
res_single = pipeline.query(single_hop)
print(f"Latency: {res_single['metrics']['total_latency']:.3f} s")
print(f"Answer: {res_single['answer']}")

print("\n=== Running Multi Hop lookup ===")
res_multi = pipeline.query(multi_hop)
print(f"Latency: {res_multi['metrics']['total_latency']:.3f} s")
print(f"Answer: {res_multi['answer']}")
print(f"Citations count: {len(res_multi['citations'])}")

### Degradation Metrics Analysis

- **Retrieval Degradation**: Multi-hop queries require a broader retrieval depth (larger K candidates). Under standard normal RAG parameters, retrieving only K=5 final chunks can cause relevant context blocks to be truncated, resulting in partial answers.
- **Latency Degradation**: Fusing multi-hop context increases token sizes, which leads to exponential scaling in query embedding, reranking, and generation latency.
- **Hallucination Risk**: When fusing facts from Apple, Nvidia, and AMD together, LLMs face higher risks of context mix-ups. The built-in contradiction detector helps prevent false claims.